In [12]:
import torch
import torch.nn as nn
torch.manual_seed(42)

In [15]:
# 1. Small dataset
X = torch.tensor([
    [[1.0], [2.0], [3.0], [4.0]],
    [[2.0], [3.0], [4.0], [5.0]],
    [[3.0], [4.0], [5.0], [6.0]],
    [[4.0], [5.0], [6.0], [7.0]],
    [[5.0], [6.0], [7.0], [8.0]],
])
y = torch.tensor([
    [5.0],
    [6.0],
    [7.0],
    [8.0],
    [9.0],
])

In [16]:
X.shape

torch.Size([5, 4, 1])

In [17]:
y.shape

torch.Size([5, 1])

In [18]:
# 2. Raw RNN implementation
class RawRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.hidden_size = hidden_size

        # Input-to-hidden parameters
        self.W_xh = nn.Parameter(torch.randn(input_size, hidden_size) * 0.1)

        # Hidden-to-hidden recurrent parameters
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1)

        # Hidden bias
        self.b_h = nn.Parameter(torch.zeros(hidden_size))

        # Hidden-to-output parameters
        self.W_hy = nn.Parameter(torch.randn(hidden_size, output_size) * 0.1)

        # Output bias
        self.b_y = nn.Parameter(torch.zeros(output_size))

    def recurrent_neuron(self, x_t, h_previous):
        """
        Computes one recurrent step.

        x_t        : current input
        h_previous : memory from the previous time step
        """

        input_component = x_t @ self.W_xh
        memory_component = h_previous @ self.W_hh

        pre_activation = (input_component + memory_component + self.b_h)

        # Hidden-state activation
        h_current = torch.tanh(pre_activation)

        return h_current

    def forward(self, x, show_steps=False):
        batch_size, sequence_length, _ = x.shape

        # Initial memory is zero
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)

        hidden_states = []

        # Manually process every time step
        for t in range(sequence_length):
            x_t = x[:, t, :]

            # Update the RNN memory
            h = self.recurrent_neuron(x_t, h)

            hidden_states.append(h)

            if show_steps:
                print(f"\nTime step t = {t + 1}")
                print("Current input x_t:")
                print(x_t)

                print("Updated hidden memory h_t:")
                print(h)

        # Use the final hidden state for prediction
        output = h @ self.W_hy + self.b_y

        # [batch_size, sequence_length, hidden_size]
        hidden_states = torch.stack(hidden_states, dim=1)

        return output, hidden_states

In [28]:
X[:,1,:]

tensor([[2.],
        [3.],
        [4.],
        [5.],
        [6.]])

In [20]:
# 3. Create model
model = RawRNN(input_size=1, hidden_size=8, output_size=1)

loss_function = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


In [22]:
# 4. Train the RNN
number_of_epochs = 2000

for epoch in range(number_of_epochs):
    # Reset gradients from the previous iteration
    optimizer.zero_grad()

    # Forward propagation through time
    prediction, hidden_states = model(X)

    # Calculate scalar loss
    loss = loss_function(prediction, y)

    # Backpropagation through time
    loss.backward()

    # Update all model parameters
    optimizer.step()

    if (epoch + 1) % 200 == 0:
        print(
            f"Epoch: {epoch + 1:4d} | "
            f"Loss: {loss.item():.6f}"
        )


Epoch:  200 | Loss: 0.024506
Epoch:  400 | Loss: 0.001588
Epoch:  600 | Loss: 0.000253
Epoch:  800 | Loss: 0.000029
Epoch: 1000 | Loss: 0.000002
Epoch: 1200 | Loss: 0.000000
Epoch: 1400 | Loss: 0.000000
Epoch: 1600 | Loss: 0.000000
Epoch: 1800 | Loss: 0.000000
Epoch: 2000 | Loss: 0.000000


In [ ]:
# 5. Test prediction
model.eval()

test_sequence = torch.tensor([[[6.0], [7.0], [8.0], [9.0]]])

with torch.no_grad():
    test_prediction, test_hidden_states = model(test_sequence, show_steps=True)

print("\nTest sequence:")
print(test_sequence.squeeze())

print("\nExpected next value: 10")
print("Predicted value    :", test_prediction.item())


Time step t = 1
Current input x_t:
tensor([[6.]])
Updated hidden memory h_t:
tensor([[ 0.9307, -0.9745,  0.9197, -0.9756,  0.9729,  0.9790, -0.9977, -0.9614]])

Time step t = 2
Current input x_t:
tensor([[7.]])
Updated hidden memory h_t:
tensor([[ 0.9982, -0.9877,  0.9977, -0.9876,  0.9946,  0.9918, -0.7119, -0.9986]])

Time step t = 3
Current input x_t:
tensor([[8.]])
Updated hidden memory h_t:
tensor([[ 0.9984, -0.9907,  0.9984, -0.9916,  0.9967,  0.9931, -0.8492, -0.9992]])

Time step t = 4
Current input x_t:
tensor([[9.]])
Updated hidden memory h_t:
tensor([[ 0.9992, -0.9960,  0.9990, -0.9962,  0.9984,  0.9974, -0.9657, -0.9996]])

Test sequence:
tensor([6., 7., 8., 9.])

Expected next value: 10
Predicted value    : 9.207667350769043


In [ ]:
# 6. Inspect how memory changes over time
print("\nHidden-state memory at every time step:")

for t in range(test_hidden_states.shape[1]):
    print(f"t = {t + 1}:",test_hidden_states[0, t])


Hidden-state memory at every time step:
t = 1: tensor([ 0.9307, -0.9745,  0.9197, -0.9756,  0.9729,  0.9790, -0.9977, -0.9614])
t = 2: tensor([ 0.9982, -0.9877,  0.9977, -0.9876,  0.9946,  0.9918, -0.7119, -0.9986])
t = 3: tensor([ 0.9984, -0.9907,  0.9984, -0.9916,  0.9967,  0.9931, -0.8492, -0.9992])
t = 4: tensor([ 0.9992, -0.9960,  0.9990, -0.9962,  0.9984,  0.9974, -0.9657, -0.9996])
